In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
df = pd.read_csv(r'C:/Users/hibah/rd/Final_SDM_Data.csv')

In [3]:

non_predictors = ['presence', 'koppen_zone', 'decimalLongitude', 'decimalLatitude']
predictor_cols = [col for col in df.columns if col not in non_predictors]

X = df[predictor_cols]
y = df['presence']

# 2. Define the VIF calculation function
def calculate_vif(data):
    vif = pd.DataFrame()
    vif["Feature"] = data.columns
    vif["VIF"] = [
        variance_inflation_factor(data.values, i)
        for i in range(data.shape[1])
    ]
    return vif

# 3. Iteratively drop features with VIF >= 5
X_vif = X.copy()
while True:
    vif = calculate_vif(X_vif)
    max_vif = vif["VIF"].max()

    if max_vif < 5:
        break

    feature_to_remove = vif.loc[vif["VIF"].idxmax(), "Feature"]
    print(f"Removing {feature_to_remove}: VIF={max_vif:.2f}")
    X_vif = X_vif.drop(columns=[feature_to_remove])

print("\nFinal Selected Variables:")
print(X_vif.columns.tolist())

Removing bio5_max_temp_warmest_month: VIF=1801439850948198.50
Removing bio10_mean_temp_warmest_quarter: VIF=56526.08
Removing bio11_mean_temp_coldest_quarter: VIF=25481.51
Removing bio1_annual_mean_temperature: VIF=5788.91
Removing bio7_temperature_annual_range: VIF=2068.39
Removing bio16_precipitation_wettest_quarter: VIF=715.18
Removing bio8_mean_temp_wettest_quarter: VIF=500.67
Removing bio2_mean_diurnal_range: VIF=217.78
Removing bio12_annual_precipitation: VIF=147.59
Removing bio9_mean_temp_driest_quarter: VIF=121.74
Removing bio15_precipitation_seasonality: VIF=97.37
Removing bio3_isothermality: VIF=59.58
Removing bio17_precipitation_driest_quarter: VIF=18.69
Removing bio6_min_temp_coldest_month: VIF=5.25

Final Selected Variables:
['dist_in_km_railways', 'dist_in_km_road', 'distance_in_km_urban', 'distance_in_km_lakes', 'dist_in_km_river', 'bio13_precipitation_wettest_month', 'bio14_precipitation_driest_month', 'bio18_precipitation_warmest_quarter', 'bio19_precipitation_coldest_

In [4]:
#split data

In [5]:
from sklearn.model_selection import train_test_split

# 1. Automatically extract the surviving feature names from your VIF loop
final_vif_predictors = X_vif.columns.tolist()
print(f"Selected {len(final_vif_predictors)} predictors after VIF filtering:\n{final_vif_predictors}")

# 2. Assign final X and y using ONLY the VIF-filtered features
X = df[final_vif_predictors]
y = df['presence']

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

Selected 10 predictors after VIF filtering:
['dist_in_km_railways', 'dist_in_km_road', 'distance_in_km_urban', 'distance_in_km_lakes', 'dist_in_km_river', 'bio13_precipitation_wettest_month', 'bio14_precipitation_driest_month', 'bio18_precipitation_warmest_quarter', 'bio19_precipitation_coldest_quarter', 'bio4_temperature_seasonality']


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_params = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

rf_grid.fit(X_train_scaled, y_train)

rf_best = rf_grid.best_estimator_

print(rf_grid.best_params_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500}


In [8]:
from sklearn.tree import DecisionTreeClassifier

dt_params = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

dt_grid.fit(X_train_scaled, y_train)

dt_best = dt_grid.best_estimator_
print("Best Parameters:", dt_grid.best_params_)

Best Parameters: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10}


In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

knn_grid.fit(X_train_scaled, y_train)

knn_best = knn_grid.best_estimator_
print("Best Parameters:", knn_grid.best_params_)

Best Parameters: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}


In [10]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

svm_params = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}

svm_grid = GridSearchCV(
    SVC(probability=True),
    svm_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

svm_grid.fit(X_train_scaled, y_train)

svm_best = svm_grid.best_estimator_

print("Best Parameters:", svm_grid.best_params_)


Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


In [11]:
from sklearn.linear_model import LogisticRegression

lr_params = {
    'C': [0.01, 0.1, 1, 10, 100]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=5000),
    lr_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

lr_grid.fit(X_train_scaled, y_train)

lr_best = lr_grid.best_estimator_
print("Best Parameters:", lr_grid.best_params_)

Best Parameters: {'C': 0.1}


In [20]:
from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1]
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ),
    xgb_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

xgb_grid.fit(X_train_scaled, y_train)

xgb_best = xgb_grid.best_estimator_
print("Best Parameters:", xgb_grid.best_params_)

Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = None

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": auc
    }

In [22]:
results = pd.DataFrame({
    "Random Forest": evaluate_model(rf_best, X_test_scaled, y_test),
    "Decision Tree": evaluate_model(dt_best, X_test_scaled, y_test),
    "KNN": evaluate_model(knn_best, X_test_scaled, y_test),
    "Logistic": evaluate_model(lr_best, X_test_scaled, y_test),
    "XGBoost": evaluate_model(xgb_best, X_test_scaled, y_test),
    "SVM": evaluate_model(svm_best, X_test_scaled, y_test)
}).T

In [27]:
results.sort_values(
    by="ROC_AUC",
    ascending=False
)

,Accuracy,Precision,Recall,F1,ROC_AUC
Random Forest,0.827321,0.833988,0.817132,0.825474,0.899156
XGBoost,0.822511,0.833002,0.806545,0.819560,0.890065
KNN,0.801828,0.783198,0.834456,0.808015,0.888849
SVM,0.780664,0.794147,0.757459,0.775369,0.857423
Decision Tree,0.756133,0.755769,0.756497,0.756133,0.816829
Logistic,0.735931,0.728119,0.752647,0.740180,0.800823
